# Implementation and test of the TDS algorithm for conditional sampling 

## Utils

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from torchvision.utils import save_image

from diffusers import DDPMPipeline, DDIMScheduler

from classifier_training import MnistCNN

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/5 done
Epoch 2/5 done
Epoch 3/5 done
Epoch 4/5 done
Epoch 5/5 done


In [2]:
# Looking for a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load diffusion model
pipe  = DDPMPipeline.from_pretrained("1aurent/ddpm-mnist")
model = pipe.unet.to(device)
model.eval()
scheduler = pipe.scheduler


# # Looking at the scheduler configuration 
# cfg = scheduler.config
# print("beta_schedule       :", cfg.beta_schedule)         
# print("beta_start          :", cfg.beta_start)             
# print("beta_end            :", cfg.beta_end)               
# print("num_train_timesteps :", cfg.num_train_timesteps)    
# print("prediction_type     :", cfg.prediction_type)        

betas = scheduler.betas.cuda()
alphas = scheduler.alphas.cuda()
alphas_cumprod = scheduler.alphas_cumprod.cuda()
T_model = len(betas)

Loading pipeline components...: 100%|██████████| 2/2 [00:00<00:00, 36.80it/s]


In [3]:
# Loading the dataset - MNIST 
transform   = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize([0.5], [0.5])])
dataset     = datasets.MNIST("./data", train=False, download=True, transform=transform)

### Utility Functions

In [4]:
def log_gaussian(x, mean, var):
    """ 
    x is a batch of K images -> tensor (K, C, H, W)
    log N(x; mean, var·I) → (K,) 
    """
    D = x.shape[1] * x.shape[2] * x.shape[3]
    diff = (x - mean).pow(2).sum(dim=(1, 2, 3))
    return -0.5 * (D * var.log() + diff / var)

## Implementing TDS

To implement the Twisted Diffusion Sampler algorithm, we first need an utility function computing $\hat{x}_\theta$ as it is called in the original article. For this, we need to use the loaded diffusion model. 

In [5]:
# Prediction of x0 given a denoised version xt
def predict_x0(xt, t_val, model=model, scheduler=scheduler):
    ab_t = scheduler.alphas_cumprod[t_val] # alpha_bar in t_val

    t_batch = torch.tensor([t_val], device="cuda").expand(xt.shape[0])
    pred_noise = model(xt, t_batch).sample  # Prediction of the noise with the model 

    x0_hat = (xt - (1.0 - ab_t).sqrt() * pred_noise) / ab_t.sqrt() # Computing the reconstruction
    return x0_hat.clamp(-1.0, 1.0), pred_noise # We return the denoised approximation but also the predicted noise

In [6]:
def twisted_diffusion_sampler(
    model,                                 # Diffusion model
    scheduler,                             # Scheduler
    log_twisting_fn,                       # log of the twisting function 
    K,                                     # Number of particles
    device='cuda'
):
    
    alphas = scheduler.alphas.to(device)
    alphas_cumprod = scheduler.alphas_cumprod.to(device)
    betas = scheduler.betas.to(device)
    T = len(betas) 
    timesteps = list(range(T - 2, -1, -1))   # T-2 -> 0

    # Initialize x_T 
    x = torch.randn(K, 1, 28, 28, device=device)
    log_w = torch.zeros(K, device=device)
   
    for t in timesteps:
        # Resample
        w_norm = (log_w - torch.logsumexp(log_w, dim=0)).exp()
        indices = torch.multinomial(w_norm, K, replacement=True) 
        x = x[indices]
        x = x.detach().requires_grad_(True)

        # Denoise prediction
        x_hat, pred_noise = predict_x0(x, t+1, model, scheduler)
        
        # Conditional score approximation
        s = - pred_noise/(1-alphas_cumprod[t+1]).sqrt() # Score
        log_p_tilde = log_twisting_fn(x_hat)
        grad_log_p_tilde = torch.autograd.grad(outputs=log_p_tilde.sum(), inputs=x)[0]
        s_tilde = s + grad_log_p_tilde # Approximation of the conditional score
       
        # Proposal
        mu = (1/alphas[t+1].sqrt()) * (x + betas[t+1]*s)
        mu_twisted = (1/alphas[t+1].sqrt()) * (x + betas[t+1]*s_tilde)
        noise = torch.randn_like(x)
        x = (mu_twisted + betas[t+1]*noise).clamp(-1, 1)

        # Update Weights
        with torch.no_grad():
            x_hat_next, pred_noise_next = predict_x0(x, t, model, scheduler)
            log_p_tilde_next = log_twisting_fn(x_hat_next) 

            log_w = log_gaussian(x, mu, betas[t+1]) + log_p_tilde_next - log_gaussian(x, mu_twisted, betas[t+1]) -log_p_tilde

        x = x.detach()

    return x, w_norm

## Testing TDS on various conditional sampling tasks

### Noisy

In [ ]:
clean = dataset[0][0].unsqueeze(0).cuda()

# Corrupt image
sigma_obs   = .5
noisy_obs   = (clean + sigma_obs * torch.randn_like(clean)).clamp(-1, 1)

# Log-likelihood function for the denoising problem 
def log_likelihood(x0_hat, observation, sigma_obs):
    """
    Gaussian observation model:  y ~ N(x₀, σ²_obs · I)
    log p̃(y | x̂₀) = -‖y - x̂₀‖² / (2 σ²_obs)    → (K,)
    """
    diff = (x0_hat - observation)                           # (K,1,28,28)
    return -diff.pow(2).sum(dim=(1, 2, 3)) / (2 * sigma_obs ** 2)

# Define twisting function (partial application of observation)
log_twist_fn = lambda x0_hat: log_likelihood(x0_hat, noisy_obs, sigma_obs)

# Run TDS
samples, weights = twisted_diffusion_sampler(model, scheduler, log_twist_fn, K=2)

best = samples[weights.argmax()].unsqueeze(0).cpu()
save_image(torch.cat([clean.cpu(), noisy_obs.cpu(), best]),
           "images/tds_noisy_result.png", nrow=3, normalize=True)

### Inpainting

In [18]:
def create_center_mask(image_shape, mask_size=12):
    """Creates a boolean mask where the center is blocked (False = unobserved)."""
    mask = torch.ones(image_shape)
    h, w = image_shape[-2:]
    cy, cx = h // 2, w // 2
    half = mask_size // 2
    mask[..., cy-half:cy+half, cx-half:cx+half] = 0
    return mask

def log_likelihood_inpainting(x_hat, observation, mask):
    return torch.sum()

# twisting function inpainting
def log_gaussian_twisting_function_inpainting(x_hat, y_observed, mask):
    """
    Computes log p(y|x_hat) ~ N(y; x_hat * mask, variance).
    Effectively forces x_hat to match y where mask is 1.
    """
    # We only care about the error in the observed region (mask == 1)
    # y_observed should already be masked (0 in unobserved regions) or we multiply by mask
    diff = (y_observed - x_hat) * mask

    # Sum of squared errors
    sse = torch.sum(diff**2, dim=[1, 2, 3])
    return -0.5 * sse


mask = create_center_mask(clean.shape, mask_size=8).to(device)
masked_obs = clean * mask # Observed image y

# Define twisting function (partial application of observation)
log_twist_fn = lambda x0_hat: log_gaussian_twisting_function_inpainting(x0_hat, masked_obs, mask)

# Run TDS
samples, weights = twisted_diffusion_sampler(model, scheduler, log_twist_fn, K=10)

best = samples[weights.argmax()].unsqueeze(0).cpu()
save_image(torch.cat([clean.cpu(), masked_obs.cpu(), best]),
           "images/tds_inpainting_result_10particles.png", nrow=3, normalize=True)

### Class-conditional

In [11]:
classifier = MnistCNN().cuda()
classifier.load_state_dict(torch.load("models/mnist_cnn.pth"))
classifier.eval()

def log_twisting_class(x_hat, class_y):
    logits = classifier(x_hat)        # (K, 10)
    log_probs = torch.log_softmax(logits, dim=-1)
    target = torch.full((x_hat.shape[0],), class_y, dtype=torch.long, device=device)
    return log_probs[torch.arange(x_hat.shape[0], device=device), target]

In [19]:
TARGET_DIGIT = 0

log_twist_fn = lambda x0_hat: log_twisting_class(x0_hat, TARGET_DIGIT)

# Run TDS
samples, weights = twisted_diffusion_sampler(model, scheduler, log_twist_fn, K=50)

best = samples[weights.argmax()].unsqueeze(0).cpu()
best_10 = samples[weights.topk(10).indices].cpu()
save_image(samples.cpu(), "images/tds_class_result.png", nrow=10, normalize=True)
